In [1]:
from ultralytics import YOLO

In [2]:
import os, random, shutil

os.makedirs("val/images", exist_ok=True)
os.makedirs("val/labels", exist_ok=True)

In [3]:
images = os.listdir("train/images")
random.shuffle(images)

split = int(len(images) * 0.8)

In [4]:
for img in images[split:]:
    shutil.move(f"train/images/{img}", f"val/images/{img}")
    label = img.rsplit(".", 1)[0] + ".txt"
    shutil.move(f"train/labels/{label}", f"val/labels/{label}")

In [5]:
with open("data.yaml", "w") as f:
    f.write("train: ./train/images\nval: ./val/images\n\nnc: 1\nnames: ['ball']")

In [8]:
import os

# Delete the corrupted file
cache_path = os.path.expanduser("~/.cache/ultralytics/yolov8n.pt")
if os.path.exists(cache_path):
    os.remove(cache_path)
    print("Deleted corrupted cache")

In [9]:
import urllib.request

urllib.request.urlretrieve(
    "https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n.pt",
    "yolov8n.pt"
)

model = YOLO("yolov8n.pt")

In [10]:
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

In [11]:
model.train(
    data="data.yaml",
    epochs=100,
    imgsz=320,
    batch=8,
    name="ball_detector"
)

New https://pypi.org/project/ultralytics/8.4.45 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ball_detecto

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001F5FD0D83D0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [12]:
model = YOLO("runs/detect/ball_detector/weights/best.pt")
model.export(format="ncnn", imgsz=320)

Ultralytics 8.4.21  Python-3.11.9 torch-2.6.0+cu124 CPU (AMD Ryzen 7 5800H with Radeon Graphics)
WARNING NCNN export does not support end2end models, disabling end2end branch.
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs\detect\ball_detector\weights\best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 5, 2100) (5.9 MB)

NCNN: starting export with NCNN 1.0.20260114 and PNNX 20260409...
NCNN: export success  5.5s, saved as 'runs\detect\ball_detector\weights\best_ncnn_model' (11.6 MB)

Export complete (5.8s)
Results saved to C:\Users\harkh\edge_goalkeeper\FULL_NEW\RANDOM.yolov8\runs\detect\ball_detector\weights
Predict:         yolo predict task=detect model=runs\detect\ball_detector\weights\best_ncnn_model imgsz=320 
Validate:        yolo val task=detect model=runs\detect\ball_detector\weights\best_ncnn_model imgsz=320 data=data.yaml  
Visualize:       https://netron.app


'runs\\detect\\ball_detector\\weights\\best_ncnn_model'

In [3]:
model = YOLO(r"C:\Users\harkh\edge_goalkeeper\FULL_NEW\RANDOM.yolov8\runs\detect\ball_detector\weights\best.pt")

# 2. Export to ONNX with a fixed image size of 160x160
model.export(format="onnx", imgsz=160)

Ultralytics 8.4.21  Python-3.11.9 torch-2.6.0+cu124 CPU (AMD Ryzen 7 5800H with Radeon Graphics)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\harkh\edge_goalkeeper\FULL_NEW\RANDOM.yolov8\runs\detect\ball_detector\weights\best.pt' with input shape (1, 3, 160, 160) BCHW and output shape(s) (1, 5, 525) (5.9 MB)

ONNX: starting export with onnx 1.21.0 opset 19...
ONNX: slimming with onnxslim 0.1.92...
ONNX: export success  1.5s, saved as 'C:\Users\harkh\edge_goalkeeper\FULL_NEW\RANDOM.yolov8\runs\detect\ball_detector\weights\best.onnx' (11.5 MB)

Export complete (1.7s)
Results saved to C:\Users\harkh\edge_goalkeeper\FULL_NEW\RANDOM.yolov8\runs\detect\ball_detector\weights
Predict:         yolo predict task=detect model=C:\Users\harkh\edge_goalkeeper\FULL_NEW\RANDOM.yolov8\runs\detect\ball_detector\weights\best.onnx imgsz=160 
Validate:        yolo val task=detect model=C:\Users\harkh\edge_goalkeeper\FULL_NEW\RANDOM.yolov8

'C:\\Users\\harkh\\edge_goalkeeper\\FULL_NEW\\RANDOM.yolov8\\runs\\detect\\ball_detector\\weights\\best.onnx'